In [120]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np
import time
import random
import torch.nn.functional as F

In [121]:
# env = gym.make('Pendulum-v1', render_mode='human')
# state = env.reset()
# print(state)

In [122]:
# env.close()

In [123]:
# # simulate the environment
# episodeNumber = 10
# timesteps = 100
# for episodeIndex in range(episodeNumber):
#     initial_state= env.reset()
#     env.render()
#     for timeIndex in range(timesteps):
#         rand_action = env.action_space.sample()
#         observation, reward, terminated, truncated, info = env.step(rand_action)
#         time.sleep(0.01)
#         if(terminated):
#             time.sleep(1) #increase to see the separate episodes
#             break

# env.close()

In [124]:
from collections import deque

class ReplayBuffer:
    def __init__(self, buffer_size=50000):
        self.buffer = deque(maxlen=buffer_size)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.stack, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [125]:
class CriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()
        self.critic1= nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )
        self.critic2 = nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, state, action):
        inp = torch.cat([state, action], dim=-1)
        Q1 = self.critic1(inp)
        Q2 = self.critic2(inp)
        return Q1, Q2

In [126]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims, action_limit):
        super().__init__()
        self.action_limit = action_limit
        self.mean_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[1], action_dim),
                )
        self.log_std_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[1], action_dim),
                )

    def forward(self, state):
        mean = self.mean_head(state)
        log_std = self.log_std_head(state)

        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        sample = dist.rsample()
        action = torch.tanh(sample) 

        log_prob = dist.log_prob(sample)
        log_prob -= torch.log(1-action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        action = action * self.action_limit
        
        return action, log_prob

In [127]:
class SoftActorCritic:
    def __init__(self):
        self.env = gym.make('Pendulum-v1')
        
        self.critic = CriticNetwork(3, 1, (64, 64))
        self.actor = ActorNetwork(3, 1, (64, 64), 2)

        self.target_critic = CriticNetwork(3, 1, (64, 64))
        self.target_critic.load_state_dict(self.critic.state_dict())
        for param in self.target_critic.parameters():
            param.requires_grad = False

        self.replay_buffer = ReplayBuffer()

        self.alpha = 0.1
        self.gamma = 0.98
        self.batch_size = 64

        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=4e-4)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=4e-4)

    @torch.no_grad()
    def get_target(self, next_states, rewards, dones):
        next_states = torch.as_tensor(next_states, dtype=torch.float32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.as_tensor(dones, dtype=torch.float32).unsqueeze(1)
        next_actions, log_prob = self.actor(next_states)

        target_Q1, target_Q2 = self.target_critic(next_states, next_actions)
        target_Q = torch.min(target_Q1, target_Q2)
        target = rewards + self.gamma * (1-dones) * (
            target_Q - self.alpha * log_prob
        )
        return target

    def update_critic(self, targets, states, actions):
        states = torch.as_tensor(states, dtype=torch.float32)
        actions = torch.as_tensor(actions, dtype=torch.float32)

        Q1, Q2 = self.critic(states, actions)

        critic_loss = F.mse_loss(Q1, targets) + F.mse_loss(Q2, targets)   

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

    def update_actor(self, states):
        states = torch.as_tensor(states, dtype=torch.float32)

        new_actions, log_prob = self.actor(states)
        Q1, Q2 = self.critic(states, new_actions)
        Q = torch.min(Q1, Q2)

        actor_loss = (self.alpha * log_prob - Q).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

    def soft_target_update(self, tau=0.005):
        for target_param, critic_param in zip(self.target_critic.parameters(), 
                                            self.critic.parameters()):
            target_param.data.copy_(
                tau * critic_param.data +
                (1-tau) * target_param.data
            )

    def train(self, n_iterations):
        self.actor.train()
        self.critic.train()
        state, _ = self.env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        episode_reward = 0
        for step in range(n_iterations):
            if step < 2000:
                action = self.env.action_space.sample()
            else:
                action, _ = self.actor(state)
                action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            episode_reward += reward
            done = terminated or truncated
            self.replay_buffer.push(state.squeeze(0).numpy(), action, reward, next_state, done)

            if len(self.replay_buffer) >= self.batch_size:
                states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
                
                targets = self.get_target(next_states, rewards, dones)
                self.update_critic(targets, states, actions)
                self.update_actor(states)
                self.soft_target_update()

            if done:    
                print(f"Step {step}: {episode_reward:.2f}")
                episode_reward = 0
                state, _ = self.env.reset()
                state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
                
            else:
                state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

    @torch.no_grad()
    def eval(self):
        self.actor.eval()
        eval_env =  gym.make("Pendulum-v1", render_mode="human")
        state, _ = eval_env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        done = False
        while(not done):
            action, _ = self.actor(state)
            action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)
            time.sleep(0.01)
        eval_env.close()

In [128]:
sac = SoftActorCritic()
sac.train(70000)
sac.eval()

Step 199: -761.50
Step 399: -1021.27
Step 599: -1569.45
Step 799: -884.75
Step 999: -1261.06
Step 1199: -1520.34
Step 1399: -1077.61
Step 1599: -1369.11
Step 1799: -971.90
Step 1999: -966.50
Step 2199: -1623.70
Step 2399: -1251.54
Step 2599: -1429.39
Step 2799: -937.90
Step 2999: -1060.14
Step 3199: -1363.98
Step 3399: -903.95
Step 3599: -886.04
Step 3799: -1132.96
Step 3999: -1359.33
Step 4199: -130.99
Step 4399: -131.19
Step 4599: -929.71
Step 4799: -798.35
Step 4999: -837.58
Step 5199: -1253.00
Step 5399: -1388.16
Step 5599: -820.47
Step 5799: -281.87
Step 5999: -714.99
Step 6199: -254.13
Step 6399: -9.97
Step 6599: -381.06
Step 6799: -514.78
Step 6999: -262.42
Step 7199: -388.52
Step 7399: -277.24
Step 7599: -304.05
Step 7799: -260.87
Step 7999: -259.25
Step 8199: -130.54
Step 8399: -495.72
Step 8599: -691.08
Step 8799: -129.82
Step 8999: -1.09
Step 9199: -260.59
Step 9399: -382.42
Step 9599: -134.26
Step 9799: -369.77
Step 9999: -362.68
Step 10199: -1.30
Step 10399: -130.26
Step 1

In [131]:
for i in range(10):
    sac.eval()

In [132]:
torch.save({
    "actor": sac.actor.state_dict(),
    "critic": sac.critic.state_dict(),
    "target_critic": sac.target_critic.state_dict(),
    "actor_optimizer": sac.actor_optimizer.state_dict(),
    "critic_optimizer": sac.critic_optimizer.state_dict(),
}, "sac_checkpoint.pt")